# Fine-Tune a Frontier Model: Natural Language → SQL

You will fine-tune a modern frontier model (e.g. **GPT-4.1-nano**) to translate plain English into SQL queries.

**Example:**
- **User:** Show all customers created last week
- **Model output:**
```sql
SELECT * FROM customers
WHERE created_at >= NOW() - INTERVAL '7 days';
```

**Steps:** build NL→SQL training pairs → export to JSONL → upload to OpenAI → create fine-tuning job → run inference with the fine-tuned model.

**Requirements:** `OPENAI_API_KEY` in `.env`; `openai` and `python-dotenv` installed.

In [8]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"
openrouter_key=os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASEURL="https://openrouter.ai/api/v1"

client = OpenAI(base_url=OPENROUTER_BASEURL, api_key=openrouter_key)
# Fine-tuning and fine-tuned model inference use OpenAI API (OpenRouter doesn't support fine-tuning)
openai_client = OpenAI()

## 1. Training data: Natural language → SQL pairs

Each example is (user question, SQL). Use PostgreSQL-style SQL for consistency (e.g. `INTERVAL '7 days'`).

In [9]:
# (natural_language_question, sql_query)
NL_SQL_PAIRS = [
    ("Show all customers created last week", "SELECT * FROM customers\nWHERE created_at >= NOW() - INTERVAL '7 days';"),
    ("List orders from the last 30 days", "SELECT * FROM orders\nWHERE order_date >= NOW() - INTERVAL '30 days';"),
    ("How many users signed up this month?", "SELECT COUNT(*) FROM users\nWHERE created_at >= date_trunc('month', NOW());"),
    ("Find the top 10 products by revenue", "SELECT product_id, SUM(quantity * unit_price) AS revenue\nFROM order_items\nGROUP BY product_id\nORDER BY revenue DESC\nLIMIT 10;"),
    ("All customers whose email contains @gmail.com", "SELECT * FROM customers\nWHERE email LIKE '%@gmail.com';"),
    ("Orders that are still pending", "SELECT * FROM orders\nWHERE status = 'pending';"),
    ("Total sales per day in January 2025", "SELECT DATE(created_at) AS day, SUM(total) AS total_sales\nFROM orders\nWHERE created_at >= '2025-01-01' AND created_at < '2025-02-01'\nGROUP BY DATE(created_at);"),
    ("Customers who have never placed an order", "SELECT c.* FROM customers c\nLEFT JOIN orders o ON c.id = o.customer_id\nWHERE o.id IS NULL;"),
    ("Average order value by customer", "SELECT customer_id, AVG(total) AS avg_order_value\nFROM orders\nGROUP BY customer_id;"),
    ("Delete all expired sessions", "DELETE FROM sessions\nWHERE expires_at < NOW();"),
]

print(f"{len(NL_SQL_PAIRS)} NL → SQL training examples")

10 NL → SQL training examples


In [10]:
def messages_for(nl_question: str, sql: str):
    """One training example in OpenAI chat format."""
    return [
        {"role": "user", "content": nl_question},
        {"role": "assistant", "content": sql.strip()},
    ]

# Preview
print(json.dumps({"messages": messages_for(NL_SQL_PAIRS[0][0], NL_SQL_PAIRS[0][1])}, indent=2))

{
  "messages": [
    {
      "role": "user",
      "content": "Show all customers created last week"
    },
    {
      "role": "assistant",
      "content": "SELECT * FROM customers\nWHERE created_at >= NOW() - INTERVAL '7 days';"
    }
  ]
}


## 2. Export to JSONL and split train/validation

In [11]:
OUT_DIR = Path("jsonl")
OUT_DIR.mkdir(exist_ok=True)

def write_jsonl(pairs, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        for nl, sql in pairs:
            row = {"messages": messages_for(nl, sql)}
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

# Use first 8 for train, last 2 for validation (or adjust)
train_pairs = NL_SQL_PAIRS[:8]
val_pairs = NL_SQL_PAIRS[8:]
write_jsonl(train_pairs, OUT_DIR / "fine_tune_train.jsonl")
write_jsonl(val_pairs, OUT_DIR / "fine_tune_validation.jsonl")
print(f"Train: {OUT_DIR / 'fine_tune_train.jsonl'} ({len(train_pairs)} examples)")
print(f"Val:   {OUT_DIR / 'fine_tune_validation.jsonl'} ({len(val_pairs)} examples)")

Train: jsonl\fine_tune_train.jsonl (8 examples)
Val:   jsonl\fine_tune_validation.jsonl (2 examples)


## 3. Upload files and create fine-tuning job

In [12]:
# Upload via direct HTTP (SDK file upload can return 405 with some envs)
import httpx

train_path = OUT_DIR / "fine_tune_train.jsonl"
val_path = OUT_DIR / "fine_tune_validation.jsonl"
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_BASE_URL", "https://api.openai.com").rstrip("/")

def upload_file(path: Path, purpose: str = "fine-tune"):
    url = f"{base_url}/v1/files"
    with open(path, "rb") as f:
        r = httpx.post(
            url,
            headers={"Authorization": f"Bearer {api_key}"},
            files={"file": (path.name, f, "application/jsonl")},
            data={"purpose": purpose},
            timeout=60.0,
        )
    r.raise_for_status()
    return r.json()

train_file = upload_file(train_path)
validation_file = upload_file(val_path)
train_file_id = train_file["id"]
validation_file_id = validation_file["id"]
print(f"Training file ID: {train_file_id}")
print(f"Validation file ID: {validation_file_id}")

Training file ID: file-LzpUyUXj3mC4nCeX4jv1pG
Validation file ID: file-7Rhnahe5egsapki9DnfPYU


In [ ]:
# Frontier model; fallback to gpt-4o-mini if your account doesn't have nano fine-tuning
BASE_MODEL = "gpt-4.1-nano-2025-04-14"
FALLBACK_MODEL = "gpt-4o-mini-2024-07-18"

# Ensure OpenAI client exists if you ran this cell without running the first cell
if "openai_client" not in globals():
    from dotenv import load_dotenv
    from openai import OpenAI
    load_dotenv(override=True)
    openai_client = OpenAI()

def create_ft_job():
    return openai_client.fine_tuning.jobs.create(
        training_file=train_file_id,
        validation_file=validation_file_id,
        model=BASE_MODEL,
        seed=42,
        hyperparameters={"n_epochs": 1},
        suffix="nl-to-sql",
    )

try:
    job = create_ft_job()
except Exception as e:
    if "model" in str(e).lower() or "404" in str(e) or "invalid" in str(e).lower():
        print(f"{BASE_MODEL} failed ({e}). Trying {FALLBACK_MODEL}...")
        job = openai_client.fine_tuning.jobs.create(
            training_file=train_file_id,
            validation_file=validation_file_id,
            model=FALLBACK_MODEL,
            seed=42,
            hyperparameters={"n_epochs": 1},
            suffix="nl-to-sql",
        )
    else:
        raise

job_id = job.id
print(f"Fine-tuning job created: {job_id}")
print("Track at: https://platform.openai.com/finetune")

## 4. Poll until job completes and get model name

In [ ]:
import time

while True:
    status = openai_client.fine_tuning.jobs.retrieve(job_id)
    print(status.status)
    if status.status == "succeeded":
        fine_tuned_model_name = status.fine_tuned_model
        print(f"Done. Model: {fine_tuned_model_name}")
        break
    if status.status == "failed":
        print(status)
        raise RuntimeError("Fine-tuning job failed")
    time.sleep(10)

## 5. Inference: ask the fine-tuned model for SQL

In [ ]:
def nl_to_sql(question: str, model: str = None) -> str:
    model = model or fine_tuned_model_name
    r = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": question}],
        max_tokens=512,
    )
    return (r.choices[0].message.content or "").strip()

In [ ]:
# Example from the intro
user_question = "Show all customers created last week"
sql_out = nl_to_sql(user_question)
print("User:", user_question)
print("Model output:")
print(sql_out)

## 6. Test and evaluate the fine-tuned model

Run the model on the validation set (and optionally all examples) and compare predicted SQL to the expected SQL. Metrics: **exact match** and **normalized match** (ignoring extra whitespace and semicolons).

In [ ]:
import re

def normalize_sql(sql: str) -> str:
    """Normalize SQL for comparison: lowercase, collapse whitespace, strip trailing semicolons."""
    if not sql:
        return ""
    s = sql.strip().lower()
    s = re.sub(r";\s*$", "", s)  # trailing semicolon
    s = re.sub(r"\s+", " ", s)    # collapse whitespace
    return s.strip()

def evaluate(pairs, model_name=None):
    """Run model on (question, expected_sql) pairs and compute accuracy."""
    results = []
    for question, expected in pairs:
        predicted = nl_to_sql(question, model=model_name)
        exact = predicted.strip() == expected.strip()
        norm = normalize_sql(predicted) == normalize_sql(expected)
        results.append({
            "question": question,
            "expected": expected.strip(),
            "predicted": predicted,
            "exact_match": exact,
            "normalized_match": norm,
        })
    return results

# Evaluate on validation set (held-out during training)
print("=== Validation set (2 held-out examples) ===\n")
val_results = evaluate(val_pairs)
for i, r in enumerate(val_results, 1):
    print(f"--- Example {i} ---")
    print(f"Question:  {r['question']}")
    print(f"Expected:  {r['expected'][:80]}{'...' if len(r['expected']) > 80 else ''}")
    print(f"Predicted: {r['predicted'][:80]}{'...' if len(r['predicted']) > 80 else ''}")
    print(f"Exact match: {r['exact_match']}  |  Normalized match: {r['normalized_match']}")
    print()

val_exact = sum(r["exact_match"] for r in val_results) / len(val_results) * 100
val_norm = sum(r["normalized_match"] for r in val_results) / len(val_results) * 100
print(f"Validation accuracy — Exact: {val_exact:.0f}%  |  Normalized: {val_norm:.0f}%")

In [ ]:
# Optional: evaluate on full set (train + val) to see in-distribution performance
print("=== Full set (all 10 examples, including training) ===\n")
all_results = evaluate(NL_SQL_PAIRS)
all_exact = sum(r["exact_match"] for r in all_results) / len(all_results) * 100
all_norm = sum(r["normalized_match"] for r in all_results) / len(all_results) * 100
print(f"Full-set accuracy — Exact: {all_exact:.0f}%  |  Normalized: {all_norm:.0f}%\n")

# Summary table
print("Summary:")
print("-" * 70)
for i, r in enumerate(all_results, 1):
    status = "✓" if r["normalized_match"] else "✗"
    print(f"  {i:2}. {status}  {r['question'][:50]}{'...' if len(r['question']) > 50 else ''}")

In [ ]:
# Ad-hoc test: try your own natural language questions
test_questions = [
    "Show all customers created last week",
    "List orders that are still pending",
    "How many users signed up this month?",
]
for q in test_questions:
    sql = nl_to_sql(q)
    print(f"Q: {q}")
    print(f"SQL: {sql}\n")

In [ ]:
# Try a few more (including ones not in the small training set)
for q in ["List orders from the last 30 days", "How many users signed up this month?"]:
    print("User:", q)
    print("SQL:", nl_to_sql(q))
    print()